In [1]:
import pandas as pd

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path("../").resolve()))

In [3]:
df_1 = pd.read_csv("../data/raw/customers_raw_a.csv")
df_2 = pd.read_csv("../data/raw/customers_raw_b.csv")
df_labels = pd.read_csv("../data/labels/column_matches.csv")

In [4]:
display(df_1)
display(df_2)
display(df_labels)

,customer_id,full_name,email,signup_date,city
0,1,Anna Müller,anna@example.com,2024-01-10,München
1,2,Max Bauer,max@example.com,2024-02-14,Augsburg
2,3,Lea Schmidt,lea@example.com,2024-03-03,Berlin
3,4,Tom Keller,tom@example.com,2024-03-20,Hamburg


,client_id,name,email_address,registration_date,location
0,11,Anna Müller,anna@example.com,2024-01-10,München
1,12,Max Bauer,max@example.com,2024-02-14,Augsburg
2,13,Lea Schmidt,lea@example.com,2024-03-03,Berlin
3,14,Tom Keller,tom@example.com,2024-03-20,Hamburg


,table_a,column_a,table_b,column_b,label
0,customers_a,customer_id,customers_b,client_id,1
1,customers_a,full_name,customers_b,name,1
2,customers_a,email,customers_b,email_address,1
3,customers_a,signup_date,customers_b,registration_date,1
4,customers_a,city,customers_b,location,1
5,customers_a,email,customers_b,location,0
6,customers_a,signup_date,customers_b,name,0
7,customers_a,customer_id,customers_b,email_address,0
8,customers_a,city,customers_b,registration_date,0


In [5]:
from src.dataset import build_pair_dataset

In [6]:
pairs = build_pair_dataset(
    df_a=df_1,
    df_b=df_2,
    labels_df=df_labels,
    table_a_name="customers_a",
    table_b_name="customers_b"
)

In [7]:
display(pairs)
print(pairs.shape)

,column_a,column_b,text_a,text_b,label
0,customer_id,client_id,"column: customer_id | values: 1, 2, 3, 4","column: client_id | values: 11, 12, 13, 14",1
1,full_name,name,"column: full_name | values: Anna Müller, Max B...","column: name | values: Anna Müller, Max Bauer,...",1
2,email,email_address,"column: email | values: anna@example.com, max@...",column: email_address | values: anna@example.c...,1
3,signup_date,registration_date,"column: signup_date | values: 2024-01-10, 2024...",column: registration_date | values: 2024-01-10...,1
4,city,location,"column: city | values: München, Augsburg, Berl...","column: location | values: München, Augsburg, ...",1
5,email,location,"column: email | values: anna@example.com, max@...","column: location | values: München, Augsburg, ...",0
6,signup_date,name,"column: signup_date | values: 2024-01-10, 2024...","column: name | values: Anna Müller, Max Bauer,...",0
7,customer_id,email_address,"column: customer_id | values: 1, 2, 3, 4",column: email_address | values: anna@example.c...,0
8,city,registration_date,"column: city | values: München, Augsburg, Berl...",column: registration_date | values: 2024-01-10...,0


(9, 5)


In [8]:
from src.evaluate import evaluate_name_baseline

In [9]:
baseline_results = evaluate_name_baseline(df_labels)
display(baseline_results)

,column_a,column_b,label,similarity,prediction
0,customer_id,client_id,1,0.500000,0
1,full_name,name,1,0.615385,1
2,email,email_address,1,0.555556,1
3,signup_date,registration_date,1,0.571429,1
4,city,location,1,0.333333,0
5,email,location,0,0.307692,0
6,signup_date,name,0,0.400000,0
7,customer_id,email_address,0,0.083333,0
8,city,registration_date,0,0.190476,0


In [10]:
import importlib
import src.dataset

importlib.reload(src.dataset)
print(dir(src.dataset))

['ColumnMatchingDataset', 'Dataset', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'build_pair_dataset', 'column_to_text', 'pd', 'torch']


In [11]:
from src.dataset import ColumnMatchingDataset
from src.train import train_model

In [12]:
dataset = ColumnMatchingDataset(pairs)

model = train_model(dataset)

Epoch 1, Loss: 2.1594
Epoch 2, Loss: 2.0305
Epoch 3, Loss: 2.0012


In [13]:
model.eval()

for i in range(len(pairs)):
    sample = pairs.iloc[i]

    pred = model([sample["text_a"]], [sample["text_b"]]).item()

    print(f"{sample['column_a']} vs {sample['column_b']}")
    print(f"Prediction: {pred:.4f} | Label: {sample['label']}")
    print("-" * 40)

customer_id vs client_id
Prediction: 0.5429 | Label: 1
----------------------------------------
full_name vs name
Prediction: 0.5492 | Label: 1
----------------------------------------
email vs email_address
Prediction: 0.5382 | Label: 1
----------------------------------------
signup_date vs registration_date
Prediction: 0.5262 | Label: 1
----------------------------------------
city vs location
Prediction: 0.5311 | Label: 1
----------------------------------------
email vs location
Prediction: 0.5286 | Label: 0
----------------------------------------
signup_date vs name
Prediction: 0.5237 | Label: 0
----------------------------------------
customer_id vs email_address
Prediction: 0.5256 | Label: 0
----------------------------------------
city vs registration_date
Prediction: 0.5182 | Label: 0
----------------------------------------


In [14]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

predictions = []
labels_true = []

model.eval()

for i in range(len(pairs)):
    sample = pairs.iloc[i]
    prob = model([sample["text_a"]], [sample["text_b"]]).item()
    pred_label = 1 if prob >= 0.5 else 0

    predictions.append(pred_label)
    labels_true.append(sample["label"])

print("Accuracy:", accuracy_score(labels_true, predictions))
print("F1:", f1_score(labels_true, predictions))
print(classification_report(labels_true, predictions))

Accuracy: 0.5555555555555556
F1: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         4
           1       0.56      1.00      0.71         5

    accuracy                           0.56         9
   macro avg       0.28      0.50      0.36         9
weighted avg       0.31      0.56      0.40         9



/Users/jannisrudloff/06_Developer/Projects/schema-matching-pytorch/.venv311/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/jannisrudloff/06_Developer/Projects/schema-matching-pytorch/.venv311/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/jannisrudloff/06_Developer/Projects/schema-matching-pytorch/.venv311/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels wi

In [15]:
import itertools

def add_negative_samples(df_a, df_b, table_a_name, table_b_name, labels_df):
    existing = set(
        zip(labels_df["column_a"], labels_df["column_b"])
    )

    negatives = []

    for col_a, col_b in itertools.product(df_a.columns, df_b.columns):
        if (col_a, col_b) not in existing:
            negatives.append({
                "table_a": table_a_name,
                "column_a": col_a,
                "table_b": table_b_name,
                "column_b": col_b,
                "label": 0
            })

    return negatives

In [17]:
import itertools

def generate_negatives(df_a, df_b, domain, labels_df):
    positives = set(
        zip(
            labels_df[
                labels_df["table_a"] == f"{domain}_a"
            ]["column_a"],
            labels_df[
                labels_df["table_b"] == f"{domain}_b"
            ]["column_b"]
        )
    )

    negatives = []

    for col_a, col_b in itertools.product(df_a.columns, df_b.columns):
        if (col_a, col_b) not in positives:
            negatives.append({
                "table_a": f"{domain}_a",
                "column_a": col_a,
                "table_b": f"{domain}_b",
                "column_b": col_b,
                "label": 0
            })

    return pd.DataFrame(negatives)

In [16]:
import pandas as pd
from pathlib import Path

from src.dataset import build_pair_dataset

raw_path = Path("../data/raw")
labels = pd.read_csv("../data/labels/column_matches.csv")

all_pairs = []

# finde alle *_a.csv Dateien
for file_a in raw_path.glob("*_a.csv"):
    domain = file_a.stem.replace("_a", "")
    file_b = raw_path / f"{domain}_b.csv"

    if not file_b.exists():
        print(f"Skipping {domain}, no matching _b file")
        continue

    print(f"Processing: {domain}")

    df_a = pd.read_csv(file_a)
    df_b = pd.read_csv(file_b)

    pairs = build_pair_dataset(
        df_a=df_a,
        df_b=df_b,
        labels_df=labels,
        table_a_name=f"{domain}_a",
        table_b_name=f"{domain}_b"
    )

    all_pairs.append(pairs)

# alles zusammenführen
all_pairs_df = pd.concat(all_pairs, ignore_index=True)

print("Total pairs:", len(all_pairs_df))
all_pairs_df.head()

Processing: customers_raw
Processing: shipments_raw
Processing: employees_raw
Processing: products_raw
Processing: payments_raw
Processing: orders_raw
Total pairs: 0


""


In [18]:
neg_df = generate_negatives(df_a, df_b, domain, labels)

labels_extended = pd.concat([labels, neg_df], ignore_index=True)

In [19]:
display(all_pairs_df)

""
